In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Default log directory is "runs" but we can specify a custom one
# This will create a directory like "runs/cifar10_experiment_1"
writer = SummaryWriter("runs/cifar10_experiment_1")

In [4]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

In [6]:
trainset = torchvision.datasets.CIFAR10(root = "./data", train = True, download = False, transform = transform)

testset = torchvision.datasets.CIFAR10(root = "./data", train = False, download = False, transform = transform)

In [7]:
train_loader = torch.utils.data.DataLoader(trainset, batch_size = 32, shuffle = True)

test_loader = torch.utils.data.DataLoader(testset, batch_size = 32, shuffle = False)

In [8]:
# CIFAR-10 classes for easy reference
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

In [9]:
# Get a batch of training images
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Create a grid of images
img_grid = torchvision.utils.make_grid(images)

# Denormalize images to display them correctly
img_grid = img_grid / 2 + 0.5  # unnormalize

# Write the grid to TensorBoard
writer.add_image('cifar10_images', img_grid)

print(f"Image batch written to TensorBoard. Shape: {images.shape}")

Image batch written to TensorBoard. Shape: torch.Size([32, 3, 32, 32])


In [10]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), # 32x32x3 -> 32x32x16
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 32x32x16 -> 16x16x16
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 16x16x16 -> 16x16x32
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # 16x16x32 -> 8x8x32
        )
        self.fc_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, 10) # 10 output classes
        )

    def forward(self, x):
        x = self.conv_stack(x)
        x = self.fc_stack(x)
        return x

model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv_stack): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [11]:
# Use the same batch of images from Step 4 as input
writer.add_graph(model, images.to(device))
print("Model graph written to TensorBoard.")

Model graph written to TensorBoard.


In [13]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- Training Loop ---
NUM_EPOCHS = 10
print("Starting training...")

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    model.train() # Set model to training mode
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data[0].to(device), data[1].to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # --- Logging Training Loss ---
    # Log the average loss for the epoch
    avg_train_loss = running_loss / len(train_loader)
    writer.add_scalar('Loss/train', avg_train_loss, epoch)

    # --- Validation Loop ---
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    test_loss = 0.0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # --- Logging Validation Metrics ---
    avg_test_loss = test_loss / len(test_loader)
    accuracy = 100 * correct / total
    writer.add_scalar('Loss/test', avg_test_loss, epoch)
    writer.add_scalar('Accuracy/test', accuracy, epoch)

    print(f'Epoch [{epoch + 1}/{NUM_EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Test Accuracy: {accuracy:.2f}%')

print('Finished Training')

Starting training...
Epoch [1/10] | Train Loss: 1.3243 | Test Loss: 1.0541 | Test Accuracy: 62.88%
Epoch [2/10] | Train Loss: 0.9332 | Test Loss: 0.9499 | Test Accuracy: 67.05%
Epoch [3/10] | Train Loss: 0.7579 | Test Loss: 0.9100 | Test Accuracy: 68.75%
Epoch [4/10] | Train Loss: 0.6105 | Test Loss: 0.8367 | Test Accuracy: 71.92%
Epoch [5/10] | Train Loss: 0.4625 | Test Loss: 0.8845 | Test Accuracy: 71.71%
Epoch [6/10] | Train Loss: 0.3330 | Test Loss: 0.9995 | Test Accuracy: 71.44%
Epoch [7/10] | Train Loss: 0.2304 | Test Loss: 1.1971 | Test Accuracy: 70.63%
Epoch [8/10] | Train Loss: 0.1577 | Test Loss: 1.3782 | Test Accuracy: 69.39%
Epoch [9/10] | Train Loss: 0.1148 | Test Loss: 1.5129 | Test Accuracy: 70.44%
Epoch [10/10] | Train Loss: 0.1014 | Test Loss: 1.6783 | Test Accuracy: 69.55%
Finished Training


In [14]:
# Get a batch of test images
dataiter = iter(test_loader)
images, labels = next(dataiter)
images_for_viz, labels_for_viz = images.to(device), labels.to(device)

# Get predictions
outputs = model(images_for_viz)
_, predicted = torch.max(outputs, 1)

# Function to create a plot
def plot_classes_preds(images, labels, preds, class_names):
    # Denormalize
    images = images / 2 + 0.5
    # Create a matplotlib figure
    fig = plt.figure(figsize=(12, 12))
    for idx in np.arange(16): # Plot 16 images
        ax = fig.add_subplot(4, 4, idx+1, xticks=[], yticks=[])
        plt.imshow(images[idx].cpu().numpy().transpose((1, 2, 0))) # Transpose from (C, H, W) to (H, W, C)
        ax.set_title("Pred: {0}\n(GT: {1})".format(
            class_names[preds[idx]], class_names[labels[idx]]),
            color=("green" if preds[idx]==labels[idx].item() else "red"))
    return fig

# Log the figure to TensorBoard
writer.add_figure('predictions_vs_actuals',
                  plot_classes_preds(images, labels, predicted.cpu(), classes),
                  global_step=0) # Log at step 0 for this specific visualization

print("Prediction visualization figure written to TensorBoard.")

# ALWAYS close the writer
writer.close()

Prediction visualization figure written to TensorBoard.
